### Analysis of E-Commerce dataset

In [7]:
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import chi2_contingency

In [11]:
db_path = 'data_source/processed/ecommerce.db'
table_name = 'ecommerce'

##### Read data from SQLite table

In [12]:
con = sqlite3.connect(db_path)
df = pd.read_sql_query(f"SELECT * from {table_name}", con)
con.close()

In [13]:
df.head()

,index,Order_Date,Time,Aging,Customer_Id,Gender,Device_Type,Customer_Login_type,Product_Category,Product,Sales,Quantity,Discount,Profit,Shipping_Cost,Order_Priority,Payment_method
0,0,2018-01-02,2023-10-13T10:56:33,8.0,37077,Female,Web,Member,Auto & Accessories,Car Media Players,140.0,1.0,0.3,46.0,4.6,Medium,credit_card
1,1,2018-07-24,2023-10-13T20:41:37,2.0,59173,Female,Web,Member,Auto & Accessories,Car Speakers,211.0,1.0,0.3,112.0,11.2,Medium,credit_card
2,2,2018-11-08,2023-10-13T08:38:49,8.0,41066,Female,Web,Member,Auto & Accessories,Car Body Covers,117.0,5.0,0.1,31.2,3.1,Critical,credit_card
3,3,2018-04-18,2023-10-13T19:28:06,7.0,50741,Female,Web,Member,Auto & Accessories,Car & Bike Care,118.0,1.0,0.3,26.2,2.6,High,credit_card
4,4,2018-08-13,2023-10-13T21:18:39,9.0,53639,Female,Web,Member,Auto & Accessories,Tyre,250.0,1.0,0.3,160.0,16.0,Critical,credit_card


## Analysis (run `python etl.py` from project root so SQLite includes transform columns)

In [14]:
# --- 1. Time-based segmentation: category mix by shipping_speed_category ---
assert "shipping_speed_category" in df.columns, "Re-run etl.py to refresh SQLite."
mix = (
    df.groupby(["shipping_speed_category", "Product_Category"])
    .size()
    .reset_index(name="n")
)
pivot = mix.pivot(index="Product_Category", columns="shipping_speed_category", values="n").fillna(0)
display(pivot.head(10))
mix.groupby("shipping_speed_category")["n"].sum().plot.bar(title="Order lines by shipping speed")
plt.ylabel("count")
plt.tight_layout()
plt.show()

# --- 2. Customer value segment: category & order priority ---
for seg in sorted(df["customer_value_segment"].dropna().unique()):
    sub = df[df["customer_value_segment"] == seg]
    print(f"\n{seg} — top categories:\n", sub["Product_Category"].value_counts().head(5))
    print(f"{seg} — Order_Priority %:\n", sub["Order_Priority"].value_counts(normalize=True).round(3))

# --- 3. Cumulative discount vs simple loyalty proxies ---
df["_od"] = pd.to_datetime(df["Order_Date"])
cust = (
    df.groupby("Customer_Id", as_index=False)
    .agg(orders=("_od", "nunique"), max_cum_disc=("cumulative_discount", "max"), aov=("Sales", "mean"))
)
cust.plot.scatter(x="max_cum_disc", y="orders", alpha=0.25, figsize=(6, 4))
plt.xlabel("max cumulative_discount (line level)")
plt.ylabel("distinct order dates")
plt.tight_layout()
plt.show()

# --- 4. Dynamic shipping vs recorded ---
print("Sum Profit (recorded):", round(df["Profit"].sum(), 2))
print("Sum profit_after_dynamic_shipping:", round(df["profit_after_dynamic_shipping"].sum(), 2))

# --- 5. Gender: top 3 categories + chi-square (top categories only) ---
top_cats = df["Product_Category"].value_counts().nlargest(15).index
sub = df[df["Product_Category"].isin(top_cats)]
ct = pd.crosstab(sub["Gender"], sub["Product_Category"])
chi2, p, dof, _ = chi2_contingency(ct)
print(f"Chi-square (Gender x top-15 categories): chi2={chi2:.2f}, p={p:.4e}, dof={dof}")
for g in sorted(sub["Gender"].dropna().unique()):
    print(f"\nTop 3 categories ({g}):\n", sub.loc[sub["Gender"] == g, "Product_Category"].value_counts().head(3))

**Takeaways (concise)**  
- **Segments:** Prioritize fulfillment for `high_value` (e.g. priority shipping on their top categories); test promotions on `low_value` high-frequency SKUs.  
- **Discount vs loyalty:** Treat correlation as associative only; use controlled experiments for causal claims.  
- **Dynamic shipping:** If `profit_after_dynamic_shipping` drops sharply, revisit priority/category weights.  
- **Gender × category:** If the chi-square p-value is small, preferences differ significantly across genders—tailor creatives and bundles to top categories per gender; if not significant, avoid over-segmenting by gender alone.